In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import h5py

from tqdm import trange
from tqdm import tqdm
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift

import brighteyes_ism.simulation.PSF_sim as sim
import brighteyes_ism.analysis.Graph_lib as gra
import brighteyes_ism.dataio.mcs as mcs

import brighteyes_flim.tools_phasor as flim
import brighteyes_flim.graph_tools as graph

from brighteyes_mcs_file import Alignment, calibrate_h5_file, show_h5_structure_html, sum_channel_applying_shifts

from s2ism import s2ism as s2
import s2ism.psf_estimator as est




## Parameters

In [ ]:
FILE_REFERENCE = r'/home/morlando1-iit.local/manuel/18052026_flim/FLIMLabs.h5'
FILE_DATA      = r'/home/morlando1-iit.local/manuel/25052026_mito_tub_flim/25052026_tomm20_pxs50_0.h5'

FILE_REFERENCE = '/mnt/DATA/Mixed Data/TestGiuse/FLIMLabs_DFD.h5'
FILE_DATA = '/mnt/DATA/Mixed Data/TestGiuse/Coumarine_DFD.h5'

DATA_KEY = "data" # usual input key: ('data', 'data_channels_extra')
CALIBRATION_PRODUCT = {"data": "spad", "data_channels_extra": "aux"}[DATA_KEY]
INSPECT_DATA_KEY = DATA_KEY

# ── Calibration parameters ─────────────────────────────────────────────────

TAU_REF              = 2.5
REFERENCE_TYPE       = "ref"
FIT_MODE             = "model_shift"
FIT_TYPE             = "likelihood"
LASER_FREQ_MHZ       = None
LASER_PERIOD_NS      = None
CHANNEL_SKEW_SOURCE  = "ref"
OVERWRITE            = True




## Calibration

In [ ]:
FILE_WITH_CALIBRATION = calibrate_h5_file(
    FILE_DATA,
    FILE_REFERENCE,
    data_key=DATA_KEY,
    reference_type=REFERENCE_TYPE,
    tau_ref=TAU_REF,
    fit_mode=FIT_MODE,
    fit_type=FIT_TYPE,
    channel_skew_type="phase_cross_correlation",
    channel_skew_source=CHANNEL_SKEW_SOURCE,
    channel_skew_fit_reference_channel=12,
    channel_skew_fit_upsampling=10,
    channel_skew_fit_apodize=False,
    period_ns=LASER_PERIOD_NS,
    overwrite=OVERWRITE,
)
print(FILE_WITH_CALIBRATION)



## Load Calibrated Data

In [ ]:
with h5py.File(FILE_WITH_CALIBRATION, "r") as hf:
    calibration = hf[f"calibration/results/{CALIBRATION_PRODUCT}"]
    metadata = hf["raw/metadata"]

    laser_freq_mhz  = float(calibration.attrs["laser_frequency_mhz"])
    laser_period_ns = float(calibration.attrs["laser_period_ns"])
    nbin            = int(metadata.attrs["time_bins"])
    pixel_size_x_um = float(metadata.attrs["pixel_size_x_um"])
    pxdwelltime     = float(metadata.attrs["pixel_dwell_time_us"])

    # data_input shape: (rep, z, y, x, t_bins, channels)
    data_input                  = hf[f"raw/{CALIBRATION_PRODUCT}"][:]
    channel_skew                = calibration["timing/channel_skew_bins"][:]
    irf_common_delay_realigned  = calibration["aligned/irf_trace"][:]
    ref_common_delay_realigned  = calibration["aligned/reference_trace"][:]

dset = np.squeeze(data_input)  # (y, x, t_bins, channels)

print(f"Using calibrated laser timing: {laser_freq_mhz:.4f} MHz ({laser_period_ns:.4f} ns)")
print(f"data_input shape: {data_input.shape}  (rep, z, y, x, t_bins, channels)")




In [ ]:
show_h5_structure_html(FILE_WITH_CALIBRATION)



In [ ]:
if False:
    irf_common_delay_realigned = Alignment.clean_irf_stack(
        irf_common_delay_realigned,
        threshold=0.3,
        window=2 / (laser_period_ns / nbin),
        time_axis=0,
        normalize=True,
    )

data_summed_no_alignment = np.sum(data_input, axis=(0, 1, 2, 3, 5))
irf_summed_no_alignment  = np.sum(irf_common_delay_realigned, axis=-1)

data_summed     = sum_channel_applying_shifts(data_input, channel_skew, axis=())[0, 0, ...]
irf_summed      = sum_channel_applying_shifts(irf_common_delay_realigned, channel_skew, axis=())
ref_summed      = sum_channel_applying_shifts(ref_common_delay_realigned, channel_skew, axis=())

print("data_summed:", data_summed.shape)
print("irf_summed:", irf_summed.shape)
print("ref_summed:", ref_summed.shape)



In [ ]:
print(laser_freq_mhz, laser_period_ns, nbin)




In [ ]:
dt = laser_period_ns/nbin # DFD bin time, ns

print(f'Excitation frequency = {laser_freq_mhz:.2f} MHz')

time = np.arange(irf_common_delay_realigned.shape[0]) * dt

print(f"Time axis (ns): {time}")



In [ ]:
plt.figure(figsize=(5, 10))

for n in range(dset.shape[-1]):
    irf_n = irf_common_delay_realigned[:, n]
    irf_norm = irf_n / (irf_n.max() + 1e-10)  # normalizza a [0, 1]
    plt.plot(time, 0.8 * irf_norm + n)

plt.yticks(np.arange(25))
plt.xlabel('time (ns)')
plt.ylabel('channel')
plt.title('Impulse Response Functions')



# FLISM image reconstruction

## s$^2$ISM

We simulate the spatial PSFs of the system.

In [ ]:
exPar = sim.simSettings()
exPar.na = 1.4   # numerical aperture
exPar.wl = 488   # excitation wavelength [nm]
exPar.gamma = 45  # parameter describing the light polarization
exPar.beta = 90  # parameter describing the light polarization
exPar.n = 1.5 # refractive index
exPar.mask_sampl = 100 # pupile plane sample points

emPar = exPar.copy()
emPar.wl = 500 # emission wavelength [nm]

grid = sim.GridParameters()
grid.Nz = 2 # number of axial planes
grid.pxsizex = pixel_size_x_um*1e3 # pixel size [nm]
grid.pxsizez = 720 # axial spacing [nm]
grid.pxpitch = 75e3 # pitch of the detector array [nm]
grid.pxdim = 50e3 # size of the pixels of the detector array [nm]
grid.N = 5 # numer of pixels per axis of the array

psf_spatial, _,_ = est.psf_estimator_from_data(dset.sum(-2), exPar, emPar, grid, z_out_of_focus = grid.pxsizez)



In [ ]:
print('Out-of-focus PSFs')
fig_1 = gra.ShowDataset(psf_spatial[0])



In [ ]:
print('In-focus PSFs')
fig_2 = gra.ShowDataset(psf_spatial[1])



We merge the temporal IRFs with the spatial PSFs in a single 4D convolution kernel.

In [ ]:
psf_irf = est.combine_psf_irf(psf_spatial, irf_common_delay_realigned)



We launch the reconstruction. Since the raw data require a significant amount of memory, we split the reconstruction into multiple batches. The overlap should be a number of pixels much larger than the size of the PSF, to avoid stitching artefacts.

If CUDA is available, choose _process='gpu'_ to speed up the computation.

In [ ]:
s2_rec = s2.batch_reconstruction(dset, psf_irf, batch_size = [301, 301], overlap = 40, max_iter = 5, process='cpu')



In [ ]:
s2_flism = s2_rec[1]



In [ ]:
intensity_map = s2_flism.sum(-1)

fig, ax = plt.subplots(figsize = (15,5))

gra.ShowImg(intensity_map, pixel_size_x_um, pxdwelltime, fig = fig, ax = ax)
ax.set_title(r's$^2$ISM')



In [ ]:
ch = 12

irf_ch    = irf_common_delay_realigned[:, ch]
raw_ch    = dset[:, :, :, ch].sum(axis=(0, 1))   # somma su x,y
recon_ch  = s2_flism.sum(axis=(0, 1))             # s2_flism non ha asse canale

# normalizza a 1 per confrontare le forme
def norm(x): return x / x.max()

plt.figure(figsize=(8, 4))
plt.plot(time, norm(irf_ch),   label='IRF ch12')
plt.plot(time, norm(raw_ch),   label='Raw ch12')
plt.plot(time, norm(recon_ch), label='s²ISM recon')
plt.xlabel('time (ns)')
plt.legend()
plt.title('Confronto IRF — Raw — s²ISM ricostruito')



In [ ]:
ch = 12

irf_ch   = irf_common_delay_realigned[:, ch]
raw_ch   = dset[:, :, :, ch].sum(axis=(0, 1))
recon_ch = s2_flism.sum(axis=(0, 1))

# Calcola lo shift dai picchi
shift_bins = int(np.argmax(raw_ch) - np.argmax(recon_ch))
print(f"Shift rilevato: {shift_bins} bin ({shift_bins * dt:.3f} ns)")

# Applica lo shift circolare sull'asse temporale
s2_flism_aligned = np.roll(s2_flism, shift_bins, axis=-1)

# Verifica visiva
def norm(x): return x / x.max()

recon_aligned_ch = s2_flism_aligned.sum(axis=(0, 1))

plt.figure(figsize=(8, 4))
plt.plot(time, norm(irf_ch),          label='IRF ch12')
plt.plot(time, norm(raw_ch),          label='Raw ch12')
plt.plot(time, norm(recon_ch),        label='s²ISM (prima)', linestyle='--', alpha=0.5)
plt.plot(time, norm(recon_aligned_ch),label='s²ISM (allineato)')
plt.xlabel('time (ns)')
plt.legend()
plt.title('Allineamento temporale s²ISM')



# Phasor analysis

In [ ]:
# Phasor diretto su s²ISM — già deconvolto dall'IRF
s2_phasor_pix = flim.calculate_phasor(s2_flism_aligned, harmonic=1)
intensity_map  = s2_flism_aligned.sum(axis=-1)



In [ ]:
phasor_map  = s2_phasor_pix
tau_m_map   = flim.calculate_tau_m(phasor_map,   dfd_freq=laser_freq_mhz * 1e6) * 1e9
tau_phi_map = flim.calculate_tau_phi(phasor_map, dfd_freq=laser_freq_mhz * 1e6) * 1e9
lifetime_map = tau_m_map

threshold = 0.05
thresholded_phasor_map    = flim.threshold_phasor(intensity_map, phasor_map, threshold)
thresholded_intensity_map = flim.threshold_intensity(intensity_map, threshold)
thresholded_tau_map, _, lifetime_mask = graph.threshold_lifetime_map(
    lifetime_map, intensity=intensity_map, threshold=threshold,
)

print(f"\nPixel validi dopo threshold: {np.sum(lifetime_mask)}")



In [ ]:
fig, ax = flim.plot_phasor(
    thresholded_phasor_map,
    quadrant="all",
    bins_2dplot=400,
    cmap="viridis",
    dfd_freq=laser_freq_mhz * 1e6,
)
ax.set_title(r"Phasor — s$^2$ISM (corrected)")

